In [2]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/MS_ML


In [3]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date
import yaml
from types import SimpleNamespace

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from get_library import get_df   # CDD Vault collection export
# from tdc.multi_pred import DTI

In [4]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
## `config.RAW_PROTEOMICS_PATH` and bare `RAW_PROTEOMICS_PATH` work.

with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')

> loaded 38 params from config/config.yaml


## 0. Imports

### 0.1. Old lib

In [5]:
%%time
## latest library straight from CDD Vault (collections AJ/AK), compound name + smiles only
if CHEMLIB_OVERWRITE:
    serac_df = (get_df(vault=7108, collections=['AK', 'AJ'], columns=['name', 'smiles','Px_validated_WT(yes/no)','Px_Ligase_dependent(yes/no)',
                                                                      'Px_NameLigase_dependent','Px_Target_dependent','Px_Target_info' ])
                .rename(columns={'name': 'compound'}))
    serac_df.to_csv(CHEMLIB_PATH,sep=',',index=False)
else:
    serac_df = pd.read_csv(CHEMLIB_PATH)
serac_df = serac_df.drop_duplicates()
serac_df['Px_validated_WT(yes/no)'] = serac_df['Px_validated_WT(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]
serac_df['Px_Ligase_dependent(yes/no)'] = serac_df['Px_Ligase_dependent(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]

serac_df.shape

CPU times: user 13.6 ms, sys: 18.1 ms, total: 31.7 ms
Wall time: 62.2 ms


(8001, 7)

In [6]:
%%time
if DFRAW_OVERWRITE:
    ## Px part 1
    df_raw_20260429, MS20260429 = fn.load_proteomics_data(
        RAW_PROTEOMICS_PATH,
        CLEAN_PROTEOMICS_PATH,
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
    )

    ## Px part 2:
    df_raw_20260520, MS20260520 = fn.load_proteomics_data(
        PX_20260520_DB,            # raw per-gene table (Database export) — was wrongly CDDVault
        PX_20260520_CDDVAULT,      # metadata table (Vault export: SMILES, Collections)
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
        mode='cddvault',           # Collections recipe (drops PROTACs), join on 'Batch Molecule-Batch ID'
        collections=['AJ', 'AK'],
    )

    ## Px part 3:
    df_raw_20260529, MS20260529 = fn.load_proteomics_data(
        PX_20260529_DB,            # raw per-gene table (Database export) — was wrongly CDDVault
        PX_20260529_CDDVAULT,      # metadata table (Vault export: SMILES, Collections)
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
        mode='cddvault',           # Collections recipe (drops PROTACs), join on 'Batch Molecule-Batch ID'
        collections=['AJ', 'AK'],
    )
    # manual formatting for 20260529 metadata:
    MS20260529 = MS20260529.rename(columns={'Molecule-Batch ID':'Batch Molecule-Batch ID','Nr. Down':'MSData - Proteomics activities: Nr. Down',"Cmpd Activity"	: "MSData - Proteomics activities: Cmpd Activity"})
    parts = MS20260529['Batch Molecule-Batch ID'].str.split('-', n=2, expand=True)
    MS20260529['Molecule Name'] = parts[0] + '-' + parts[1]   # 'SRB-0000385'
    MS20260529['batch']    = parts[2]                    # '001'

    ## Making final df_raw
    df_raw = pd.concat([df_raw_20260429,df_raw_20260520,df_raw_20260529]).reset_index(drop=True)
    df_raw[['genes']].drop_duplicates().to_csv('data/MS/Px_genes.csv')

    ## check whether all compounds are including in the df_raw:

    colskeep = ['Molecule Name','MSData - Proteomics activities: Nr. Down','origin',"MSData - Proteomics activities: Cmpd Activity"]
    MS = pd.concat([MS20260429.assign(origin='MS20260429'),
                    MS20260520.assign(origin='MS20260520'),
                    MS20260529.assign(origin='MS20260529')]).reset_index(drop=True)[colskeep].rename(columns={'Molecule Name':'compound',
                                                                                                            'MSData - Proteomics activities: Nr. Down':'ndown',
                                                                                                            "MSData - Proteomics activities: Cmpd Activity":'activity'})
    MS['date'] = pd.to_datetime(MS['origin'].str.replace('MS', ''))
    
    MS.to_csv(MS_PATH,sep=',',index=False)
    df_raw.to_csv(DFRAW_PATH,sep=',',index=False)

    del df_raw_20260429,df_raw_20260520,df_raw_20260529
else:
    df_raw = pd.read_csv(DFRAW_PATH)
    MS     = pd.read_csv(MS_PATH)


CPU times: user 21.5 s, sys: 7.19 s, total: 28.7 s
Wall time: 28.7 s


In [7]:
## OpenTargets disease scores for every gene seen in the proteomics data.
## Cached to parquet — re-run is instant. Delete the file (or change OT_CACHE)
## to force a recompute from the local bulk dump under data/external/opentarget/.

if os.path.exists(OT_CACHE):
    print(f'> loading cached {OT_CACHE}')
    ot_df = pd.read_parquet(OT_CACHE)
else:
    # normalise the gene column: explode multi-gene strings (peptides mapping to >1
    # protein), strip whitespace, drop empties / NaNs, dedupe.
    unique_genes = pd.DataFrame({
        'gene': (df_raw['genes'].dropna().astype(str)
                  .str.split(r'[;,|]', regex=True).explode()
                  .str.strip()
                  .replace('', pd.NA).dropna()
                  .unique())
    })
    print(f'> {len(unique_genes):,} unique gene symbols in df_raw')

    ot_df = fn.get_opentarget_disease_score(
        unique_genes, gene_col='gene',
        top_n=20,
        ot_root=OT_ROOT,
    )
    os.makedirs(os.path.dirname(OT_CACHE), exist_ok=True)
    ot_df.to_parquet(OT_CACHE, index=False)
    print(f'> wrote {OT_CACHE}')

print(f'> {len(ot_df):,} (target, disease) rows  /  '
      f'{ot_df["target_symbol"].nunique():,} targets resolved')
ot_df.head(2)

## disease areas of interest to big pharma:
PRIORITY = {
    # Annotations: flagship products per pharma, just to anchor priority intuition.
    'cancer or benign tumor',                       # universal — BMS Opdivo/Yervoy, Roche Herceptin/Tecentriq/Phesgo, NVS Kisqali/Kymriah/Pluvicto, Pfizer Ibrance/Padcev, Lilly Verzenio/Jaypirca
    'hematologic disease',                          # BMS Revlimid/Pomalyst (multiple myeloma), Pfizer Elrexfio, NVS Tasigna, Roche Hemlibra (haemophilia), Lilly Jaypirca
    'cardiovascular disease',                       # BMS Eliquis, NVS Entresto/Leqvio (PCSK9 siRNA), Pfizer Vyndaqel (TTR amyloidosis)
    'immune system disease',                        # BMS Zeposia/Sotyktu/Orencia, NVS Cosentyx/Xolair, Pfizer Xeljanz/Cibinqo, Lilly Olumiant/Taltz, Roche Actemra
    'musculoskeletal or connective tissue disease', # RA / lupus / psoriasis (overlaps immune) — BMS Orencia, NVS Cosentyx, Pfizer Xeljanz, Lilly Olumiant
    'nervous system disease',                       # NVS Kesimpta/Gilenya (MS), Roche Ocrevus/Evrysdi (MS, SMA), Lilly Kisunla (AD, donanemab), BMS Cobenfy (post-Karuna)
    'psychiatric disorder',                         # BMS Cobenfy (schizophrenia), Lilly historic SSRIs/atypicals, otherwise smaller pharma footprint
    'nutritional or metabolic disease',             # Lilly #1 (Mounjaro / Zepbound — obesity/T2D), Pfizer & NVS chasing GLP-1 follow-ons
    'endocrine system disease',                     # Lilly (hormone/metabolic overlap), oncology adjacency (hormone-driven cancers) for BMS/Roche
    'disorder of visual system',                  # ← uncomment for Roche specifically (Vabysmo/Lucentis ophthalmology franchise)
    'respiratory or thoracic disease',            # ← uncomment for COPD/asthma plays (NVS Xolair, GSK; not named-four core)
    'infectious disease',                         # ← uncomment for vaccines/antivirals (Pfizer Comirnaty/Paxlovid/Prevnar)
}
DROP = {'phenotype', 'measurement', 'biological_process',
        'animal disease', 'medical procedure'}

def is_pharma_relevant(ta_str):
    """Keep rows whose therapeutic_areas overlap PRIORITY and aren't pure DROP-only."""
    if not isinstance(ta_str, str) or not ta_str:
        return False
    tokens = set(ta_str.split('|'))
    return bool(tokens & PRIORITY) and bool(tokens - DROP)

ot_pharma = ot_df[ot_df['therapeutic_areas'].apply(is_pharma_relevant)].reset_index(drop=True)
print(f'> {len(ot_pharma):,} / {len(ot_df):,} (target, disease) rows pass the BMS-style filter')
print(f'> covers {ot_pharma["target_symbol"].nunique():,} unique targets')

# breakdown by which priority area each row hits (a row can hit multiple)
print('\nrows per priority area:')
for token in sorted(PRIORITY):
    n = ot_pharma['therapeutic_areas'].str.split('|').apply(lambda s: token in s).sum()
    print(f'  {n:>6,}  {token}')

ot_pharma.head(2)

## get target list
ot_df_rank = (ot_pharma.sort_values('overall_score',ascending=False).reset_index(drop=True).groupby('target_symbol').first().sort_values('overall_score',ascending=False).reset_index())
target_list = list(ot_df_rank[ot_df_rank['overall_score'] > 0.4]['target_symbol'])

## add pharma target lists: BOTH the small-molecule patent file (gene col) AND
## the BMS pipeline file (hgnc_symbol col), so every pharma/BMS target is included.
patents = list(pd.read_csv(PHARMA_PATENT_CSV)['gene'].dropna().unique())
bms     = list(pd.read_csv(BMS_GENES)['hgnc_symbol'].dropna().unique())
target_list += patents + bms

target_list = list(dict.fromkeys(target_list))   # de-dupe, keep order
print(f'> target_list: {len(target_list)} unique '
      f'(OT>0.4 + {len(set(patents))} patent + {len(set(bms))} BMS genes)')
len(target_list)

> loading cached output/MS/opentargets_target_disease.parquet
> 232,004 (target, disease) rows  /  11,584 targets resolved
> 78,935 / 232,004 (target, disease) rows pass the BMS-style filter
> covers 9,105 unique targets

rows per priority area:
  17,071  cancer or benign tumor
   3,890  cardiovascular disease
   2,504  disorder of visual system
   3,466  endocrine system disease
   3,416  hematologic disease
   3,938  immune system disease
   3,710  infectious disease
   8,708  musculoskeletal or connective tissue disease
  25,393  nervous system disease
   9,849  nutritional or metabolic disease
   1,704  psychiatric disorder
   2,357  respiratory or thoracic disease
> target_list: 6706 unique (OT>0.4 + 172 patent + 46 BMS genes)


6706

In [8]:
## derive MS score
df_raw = df_raw.dropna()
df_raw['-log10(p-value)'] = -np.log10(df_raw['pvalue'])
df_raw['ms_score'] = (-df_raw['-log10(p-value)'] * df_raw['logfc']).clip(lower=0.0, upper=100.0)
df_raw = df_raw.sort_values('ms_score',ascending=False)

df_ms = df_raw[df_raw['significant']==1].groupby(['genes','MSPlate']).first().reset_index()

#### 0.2. New lib

In [9]:
FBX_MEASURE  = pd.read_csv('data/advanteidge/20260601_FBX_MEASURE.csv')
FBX_MSSCORE  = pd.read_csv('data/advanteidge/20260601_FBX_MSSCORE.csv')
FBX_REPORT   = pd.read_csv('data/advanteidge/20260601_FBX_REPORT.csv')
target2R2_df = pd.read_csv(GENE_SAR_OUT).rename(columns={'gene':'genes'})

In [10]:
## remove plate and keep max ms_score associate with target
# FBX_MSSCORE = (FBX_MSSCORE[~FBX_MSSCORE['plate'].isin(['Plate12', 'Plate15', 'Plate23'])]
#                .sort_values('ms_score_percent',ascending=False)
#                .reset_index(drop=True))
# FBX_MSSCORE = FBX_MSSCORE.groupby('genes').head(1).reset_index(drop=True)[['uniquecontrast','genes','ms_score','association_score']]

## add R2 value, note that some genes will be missing due to assocation score < 0.4 ctf
# FBX_MSSCORE = pd.merge(FBX_MSSCORE,target2R2_df[['genes','R2','pearson_r','pearson_p','spearman_r','spearman_p']]).sort_values('R2',ascending=False)

In [11]:
# ## getting the compound info, plate and activity level
# parts = FBX_REPORT['srbnumber'].str.split('-', n=2, expand=True)
# FBX_REPORT['Molecule Name'] = parts[0] + '-' + parts[1]   # 'SRB-0000385'
# FBX_REPORT['batch']    = parts[2]                    # '001'
# print(FBX_REPORT['activity'].unique())
# FBX_REPORT.head(1)


#### 0.3 combine all datasets:

In [12]:
## 1. combine df_raw & FBX_MEASURE
## Unified per-(compound x gene x experiment) MEASURE table for the whole Px set.
## df_raw (broad: 2,237 compounds x 86 plates x 12k genes) UNION FBX_MEASURE (the
## WT/KO 'Pw' condition plates). They overlap on 3 plates / 93 uniquecontrasts where
## logfc is the SAME measurement (corr 0.85, 94% identical) -> FBX kept as source of
## truth there: take ALL FBX rows + df_raw rows from experiments NOT in FBX.
_cols = ['compound', 'genes', 'pg', 'plate', 'uniquecontrast',
         'logfc', 'pvalue', 'adjpval', 'significant']

# FBX_MEASURE has no `compound` -> add it via FBX_REPORT (uniquecontrast -> srbnumber, strip batch)
_rep = FBX_REPORT[['uniquecontrast', 'srbnumber']].drop_duplicates('uniquecontrast')
_p   = _rep['srbnumber'].astype(str).str.split('-', n=2, expand=True)
_rep = _rep.assign(compound=_p[0] + '-' + _p[1])                      # SRB-XXXXXXX
fbx_std = (FBX_MEASURE.merge(_rep[['uniquecontrast', 'compound']], on='uniquecontrast', how='left')
           .reindex(columns=_cols).assign(source='FBX'))

# df_raw -> same schema (MSPlate -> plate)
dr_std = df_raw.rename(columns={'MSPlate': 'plate'}).reindex(columns=_cols).assign(source='df_raw')

# FBX is source of truth on the shared experiments
_shared_uc = set(FBX_MEASURE['uniquecontrast']) & set(df_raw['uniquecontrast'].astype(str))
measure = pd.concat([fbx_std, dr_std[~dr_std['uniquecontrast'].astype(str).isin(_shared_uc)]],
                    ignore_index=True)

print(f'> combined MEASURE: {len(measure):,} rows')
print(f'  experiments (uniquecontrast): {measure["uniquecontrast"].nunique():,} '
      f'(FBX {fbx_std["uniquecontrast"].nunique()}, df_raw-only '
      f'{dr_std["uniquecontrast"].nunique() - len(_shared_uc)}, shared->FBX {len(_shared_uc)})')
print(f'  compounds {measure["compound"].nunique():,} | genes {measure["genes"].nunique():,} '
      f'| plates {measure["plate"].nunique():,}')
print(measure["source"].value_counts().to_string())
_no_cmp = int(fbx_std["compound"].isna().sum())
if _no_cmp:
    print(f'  note: {_no_cmp:,} FBX rows have no compound (uniquecontrast absent from FBX_REPORT '
          f'- control/QC contrasts); kept for volcanoes, excluded from the compound panel.')


> combined MEASURE: 26,116,366 rows
  experiments (uniquecontrast): 2,714 (FBX 305, df_raw-only 2409, shared->FBX 93)
  compounds 2,238 | genes 12,003 | plates 93
source
df_raw    23548861
FBX        2567505
  note: 8,425 FBX rows have no compound (uniquecontrast absent from FBX_REPORT - control/QC contrasts); kept for volcanoes, excluded from the compound panel.


In [13]:
## 2. combine FBX_MSSCORE & df_ms:
## Unified per-(gene x plate) MS-SCORE table (feeds the dots). df_ms is the df_raw
## side (strongest-ms_score significant compound per gene-plate); FBX_MSSCORE is the
## curated side (+ OpenTargets association/genetic/literature scores). Same overlap
## rule as the measure combine: FBX is source of truth on shared (gene, plate).
## NB: re-read FBX_MSSCORE fresh from CSV so this cell is independent of the in-place
## edits above (the per-gene collapse drops 'plate').
_FBX_MS = pd.read_csv('data/advanteidge/20260601_FBX_MSSCORE.csv')
_FBX_MS = _FBX_MS[~_FBX_MS['plate'].isin(['Plate12', 'Plate15', 'Plate23'])]   # same noisy-plate drop
_cols = ['genes', 'plate', 'uniquecontrast', 'compound', 'pg', 'ms_score',
         'association_score', 'genetic_score', 'literature_score', 'activity',
         'logfc', 'pvalue', 'significant']

# FBX_MSSCORE -> one row per (gene, plate) [strongest ms_score], + compound via FBX_REPORT
_rep = FBX_REPORT[['uniquecontrast', 'srbnumber']].drop_duplicates('uniquecontrast')
_p   = _rep['srbnumber'].astype(str).str.split('-', n=2, expand=True)
_rep = _rep.assign(compound=_p[0] + '-' + _p[1])
fbx_gp = (_FBX_MS.sort_values('ms_score', ascending=False)
          .groupby(['genes', 'plate'], as_index=False).first()
          .merge(_rep[['uniquecontrast', 'compound']], on='uniquecontrast', how='left')
          .reindex(columns=_cols).assign(source='FBX'))

# df_ms is already per (gene, plate) (MSPlate -> plate)
df_ms_std = df_ms.rename(columns={'MSPlate': 'plate'}).reindex(columns=_cols).assign(source='df_raw')

# FBX source of truth on shared (gene, plate)
_fbx_keys = set(map(tuple, fbx_gp[['genes', 'plate']].itertuples(index=False)))
_keep = ~df_ms_std.set_index(['genes', 'plate']).index.isin(_fbx_keys)
mscore = pd.concat([fbx_gp, df_ms_std[_keep]], ignore_index=True)

print(f'> combined MS-SCORE: {len(mscore):,} (gene,plate) rows '
      f'(FBX {len(fbx_gp):,}, df_raw {int(_keep.sum()):,})')
print(f'  genes {mscore["genes"].nunique():,} | plates {mscore["plate"].nunique():,}')
print(mscore["source"].value_counts().to_string())
print(f'  association/genetic/literature present for {int(mscore["association_score"].notna().sum()):,} rows '
      f'(FBX genes only; df_raw genes get these from OpenTargets in the next step).')
mscore.head(2)


> combined MS-SCORE: 37,145 (gene,plate) rows (FBX 312, df_raw 36,833)
  genes 8,520 | plates 86
source
df_raw    36833
FBX         312
  association/genetic/literature present for 301 rows (FBX genes only; df_raw genes get these from OpenTargets in the next step).


,genes,plate,uniquecontrast,compound,pg,ms_score,association_score,genetic_score,literature_score,activity,logfc,pvalue,significant,source
0,AADAC,Pw63,SRB.0005857.001_vs_SRB.0005857.001_complement_...,SRB-0005857,P22760,5.808102,0.352390,0.579656,0.545862,Low (2-10),NaN,NaN,NaN,FBX
1,ABCA8,Pw63,SRB.0005850.001_vs_SRB.0005850.001_complement_...,SRB-0005850,O94911;O94911-3,6.402353,0.570717,0.938786,0.634522,Low (2-10),NaN,NaN,NaN,FBX


In [14]:
## 3. combine MS & FBX_REPORT  ->  source-derived per-experiment REPORT
## MS is just the compound-level collapse of CLEAN + PX_20260520 + PX_20260529, so we
## derive the df_raw-side REPORT straight from those source tables at the per-EXPERIMENT
## grain -> real Concentration (uM) + per-plate activity / nr_down / cell_line / condition.
## Join to df_raw's uniquecontrast via (MoleculeBatchID, MSPlate) [99.7% match]; MS is
## only a fallback for activity on the rare unmatched experiments. Then union with
## FBX_REPORT, FBX source of truth on shared uniquecontrasts.
P = 'MSData - Proteomics activities: '
def _load_src(path, prefixed):
    pre = P if prefixed else ''
    m = {('Batch Molecule-Batch ID' if prefixed else 'Molecule-Batch ID'): 'batch',
         pre + 'MSPlate': 'plate',
         (P + 'Concentration (uM)' if prefixed else 'Concentration'): 'concentration',
         pre + 'Cmpd Activity': 'activity', pre + 'Nr. Down': 'nr_down',
         pre + 'Cell line': 'cell_line', pre + 'Sample Condition': 'condition'}
    cols = pd.read_csv(path, nrows=0).columns
    m = {k: v for k, v in m.items() if k in cols}
    return pd.read_csv(path, usecols=list(m)).rename(columns=m)

SRC = pd.concat([_load_src(CLEAN_PROTEOMICS_PATH, True),
                 _load_src(PX_20260520_CDDVAULT, True),
                 _load_src(PX_20260529_CDDVAULT, False)], ignore_index=True)
SRC = (SRC.dropna(subset=['batch', 'plate']).drop_duplicates(['batch', 'plate'])
       .rename(columns={'batch': 'MoleculeBatchID', 'plate': 'MSPlate'}))

_cols = ['uniquecontrast', 'compound', 'plate', 'concentration', 'activity',
         'nr_down', 'cell_line', 'condition']

# df_raw side: experiments joined to source metadata on (MoleculeBatchID, MSPlate)
rep_dr = (df_raw[['uniquecontrast', 'MoleculeBatchID', 'MSPlate', 'compound']].drop_duplicates('uniquecontrast')
          .merge(SRC, on=['MoleculeBatchID', 'MSPlate'], how='left')
          .rename(columns={'MSPlate': 'plate'}))
# MS fallback for the rare unmatched activities (compound-level, latest tranche)
_msact = MS.sort_values('date').drop_duplicates('compound', keep='last').set_index('compound')['activity']
rep_dr['activity'] = rep_dr['activity'].fillna(rep_dr['compound'].map(_msact))
rep_dr = rep_dr.reindex(columns=_cols).assign(source='df_raw')

# FBX side: srbnumber -> compound (strip batch)
_rf = FBX_REPORT.copy()
_p  = _rf['srbnumber'].astype(str).str.split('-', n=2, expand=True)
_rf['compound'] = _p[0] + '-' + _p[1]
rep_fbx = _rf.reindex(columns=_cols).drop_duplicates('uniquecontrast').assign(source='FBX')

# FBX source of truth on shared uniquecontrasts
_shared_uc = set(FBX_REPORT['uniquecontrast']) & set(df_raw['uniquecontrast'].astype(str))
report = pd.concat([rep_fbx, rep_dr[~rep_dr['uniquecontrast'].astype(str).isin(_shared_uc)]],
                   ignore_index=True)

print(f'> combined REPORT: {len(report):,} uniquecontrasts | compounds {report["compound"].nunique():,} '
      f'| plates {report["plate"].nunique():,}')
print(report['source'].value_counts().to_string())
print(f'  concentration present: {100 * report["concentration"].notna().mean():.0f}% '
      f'| activity present: {100 * report["activity"].notna().mean():.0f}%')
print('  activity:', report['activity'].value_counts(dropna=False).to_dict())
report.head(2)


> combined REPORT: 2,714 uniquecontrasts | compounds 2,238 | plates 93
source
df_raw    2409
FBX        305
  concentration present: 100% | activity present: 100%
  activity: {'Low (2-10)': 1185, 'Silent': 610, 'Single (1)': 420, 'Medium (11-25)': 260, 'High (>25)': 239}


,uniquecontrast,compound,plate,concentration,activity,nr_down,cell_line,condition,source
0,SRB.0005840.001_vs_SRB.0005840.001_complement_...,SRB-0005840,Pw81BCDEKO,10.0,Medium (11-25),11.0,HepG2,WildType,FBX
1,SRB.0005518.001_vs_SRB.0005518.001_complement_...,SRB-0005518,Pw105VMMLN,10.0,Low (2-10),8.0,HepG2,WildType,FBX


In [15]:
## local params:
control_compounds = ['SRB-0000692','SRB-0000615']

In [16]:
## contaminants compounds to remove:
contaminants = list(pd.read_csv('data/MS/20260611_contaminants.csv.csv')['Molecule Name'])

In [17]:
## extract list of validated and devalidated targets
l1 = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==0) & 
                       (serac_df['Px_Target_dependent'].notnull())]['Px_Target_dependent']))
devalidated_targets = list(set([s.split(' ')[0].upper() for x in l1 for s in x.split(';')]))

l2 = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==1) & 
                       (serac_df['Px_Target_dependent'].notnull())]['Px_Target_dependent']))
validated_targets = list(set([s.split(' ')[0].upper() for x in l2 for s in x.split(';')]))


In [18]:
## degradation research per target (one record/gene): therapeutic rationale,
## degrader feasibility, DepMap dependency, indications, safety, sources, ...
import json as _json
with open('/mnt/c/Users/gtamo/Desktop/GT/disease_association/20260614_full_genome_degradation_research.json') as _f:
    GENE_RESEARCH = _json.load(_f)
print(f'> loaded degradation research for {len(GENE_RESEARCH)} genes; '
      f'fields: {list(GENE_RESEARCH[0].keys())}')

## define

> loaded degradation research for 6706 genes; fields: ['gene_name', 'target_class', 'lof_therapeutic_benefit', 'degrader_vs_inhibitor_rationale', 'degrader_feasibility', 'depmap_dependency', 'opentargets_top_indications', 'existing_degraders', 'safety_flags', 'confidence', 'biology_rationale', 'sources']


In [19]:
# ## Re-(Compute) Volcanoes
# # A p-value of 0.0 becomes +inf under -log10 and is currently floored at 1e-300 inside
# # the volcano renderers (so it plots at y=300). Instead floor every 0.0 p-value to the
# # smallest NON-zero p-value observed, then regenerate ONLY the cached volcanoes of
# # experiments (uniquecontrasts) that had >=1 floored target. The volcano disk cache is
# # keyed by identity + render params (NOT by the data), so the affected images must be
# # overwritten explicitly; step 4 then picks them up as cache hits.
# #
# # Crash-safe: `measure` is floored IN PLACE only AFTER the re-render succeeds, so a
# # failed render leaves the zeros intact and the cell stays re-runnable. (If you ever
# # half-run it, just re-run the cell that builds `measure` to restore the originals.)

# # These MUST match the plot_3d_interface call in step 4 (cache filenames depend on them).
# VOLCANO_DIR     = os.path.join(DROPBOX_ML, 'interfaces', 'volcanoes_px')
# VOLCANO_XLIM    = (-8, 8)
# VOLCANO_SIZE_PX = 350
# DROP_PLATES     = ['Plate12', 'Plate15', 'Plate23']

# _was_zero = measure['pvalue'].eq(0.0)
# if not _was_zero.any():
#     print('> no zero p-values in measure — nothing to floor or recompute '
#           '(already floored? re-run the cell that builds `measure` to restore them)')
# else:
#     # 1) smallest non-zero p-value -> the floor value
#     pmin = measure.loc[measure['pvalue'] > 0, 'pvalue'].min()
#     assert pd.notna(pmin), 'no non-zero p-values to floor to'
#     affected_uc = set(measure.loc[_was_zero, 'uniquecontrast'].unique())
#     print(f'> {int(_was_zero.sum()):,} zero p-values across {len(affected_uc):,} '
#           f'experiments -> floor to pmin={pmin:.3e}  (-log10 = {-np.log10(pmin):.1f}, '
#           f'was capped at 300)')

#     # 2) render from a FLOORED COPY (don't touch `measure` yet). Drop noisy plates like
#     #    step 4; volcanoes that exist = significant-down hits (same rule step 4 uses).
#     meas = measure[~measure['plate'].isin(DROP_PLATES)].copy()
#     meas.loc[meas['pvalue'].eq(0.0), 'pvalue'] = pmin
#     _pairs = (meas[(meas['significant'] == 1) & (meas['logfc'] < 0)
#                    & meas['uniquecontrast'].isin(affected_uc)]
#               [['genes', 'uniquecontrast']].dropna().drop_duplicates())
#     print(f'> recomputing {len(_pairs):,} (gene x experiment) volcanoes across '
#           f'{_pairs["uniquecontrast"].nunique():,} affected experiments...')
#     _vstats = fn.recompute_volcanoes(
#         meas, _pairs.itertuples(index=False, name=None), VOLCANO_DIR,
#         volcano_key='uniquecontrast', significant=True,
#         xlim=VOLCANO_XLIM, size_px=VOLCANO_SIZE_PX, n_jobs=8)

#     # 3) re-render succeeded -> NOW floor `measure` in place so step 4's volcano source
#     #    (and any downstream use) matches the regenerated images.
#     measure.loc[_was_zero, 'pvalue'] = pmin
#     print('> measure floored in place; re-run step 4 to rebuild the interface')


In [20]:
## step 4: per-gene DOTS + compounds + volcano source for the WHOLE Px set, then render.
## Built from the unified tables (measure / mscore / report), same plot_3d_interface as the
## FBX interface. Dots = one per gene: x=R2 (SAR full-genome), y=association (OpenTargets),
## z=ms_score (max over plates from `mscore`). Volcano source = `measure`. Compounds from
## `measure` significant-down hits + `report` (compound/plate/activity/concentration) + smiles.
DROP_PLATES = ['Plate12', 'Plate15', 'Plate23']

# --- volcano source = unified measure (drop noisy plates) ---
meas = measure[~measure['plate'].isin(DROP_PLATES)]

# --- gene-level axes ---
# x: SAR predictability R2 (full genome)
R2_df = (pd.read_csv('output/MS/20260529_geneSAR_R2_full_genome.csv')[['gene', 'R2']]
         .rename(columns={'gene': 'genes'}))
# y + colour: OpenTargets association (max overall_score per gene) + top disease area
if 'ot_df' not in globals():
    ot_df = pd.read_parquet(OT_CACHE)
assoc = ot_df.groupby('target_symbol')['overall_score'].max().rename('association_score')
PRIORITY = ['cancer or benign tumor', 'hematologic disease', 'cardiovascular disease',
            'immune system disease', 'musculoskeletal or connective tissue disease',
            'nervous system disease', 'psychiatric disorder',
            'nutritional or metabolic disease', 'endocrine system disease']
_rank = {a: i for i, a in enumerate(PRIORITY)}
_areas = (ot_df[['target_symbol', 'overall_score', 'therapeutic_areas']]
          .assign(area=lambda d: d['therapeutic_areas'].fillna('').str.split('|')).explode('area'))
_areas = _areas[_areas['area'].isin(_rank)].copy()
_areas['_rank'] = _areas['area'].map(_rank)
_top_area = (_areas.sort_values(['target_symbol', '_rank', 'overall_score'], ascending=[True, True, False])
             .drop_duplicates('target_symbol', keep='first')
             .rename(columns={'target_symbol': 'gene', 'area': 'disease_area'})[['gene', 'disease_area']])
# z: ms_score per gene = max over plates (from the unified mscore)
ms_gene = mscore.groupby('genes')['ms_score'].max().rename('ms_score')

# --- dots: one per gene (need R2 for the x-axis) ---
iface_df = (ms_gene.reset_index()
            .merge(R2_df, on='genes', how='inner')
            .merge(assoc, left_on='genes', right_index=True, how='left')
            .rename(columns={'genes': 'gene'})
            .merge(_top_area, on='gene', how='left'))
iface_df['association_score'] = iface_df['association_score'].fillna(0.0)
_pharma = set(pd.read_csv(PHARMA_PATENT_CSV)['gene'].dropna().unique())
_bms    = set(pd.read_csv(BMS_GENES)['hgnc_symbol'].dropna().unique())
_model  = iface_df['R2'] > PHARMA_R2_CUTOFF
iface_df.loc[iface_df['gene'].isin(_pharma) & _model, 'disease_area'] = 'pharma'
iface_df.loc[iface_df['gene'].isin(_bms)    & _model, 'disease_area'] = 'BMS'
print(f'> iface_df: {len(iface_df):,} gene dots (whole Px) | disease_area set for {iface_df["disease_area"].notna().sum():,}')

# --- compounds_df: significant-down hits from measure + report metadata + smiles ---
chemlib = pd.read_csv(CHEMLIB_PATH)[['compound', 'smiles']].drop_duplicates('compound')
n_genes = meas.dropna(subset=['logfc', 'pvalue']).groupby('uniquecontrast')['genes'].nunique().rename('n_genes')
rep = report[['uniquecontrast', 'compound', 'plate', 'concentration', 'activity']].drop_duplicates('uniquecontrast')
_hit = meas.loc[(meas['significant'] == 1) & (meas['logfc'] < 0), ['genes', 'uniquecontrast', 'logfc', 'pvalue']]
hits = _hit.merge(rep, on='uniquecontrast', how='left').rename(columns={'genes': 'gene'})
hits = hits[hits['compound'].notna() & hits['compound'].str.startswith('SRB-')]
hits = hits.sort_values(['gene', 'compound', 'plate', 'logfc'])
compounds_df = (hits.groupby(['gene', 'compound', 'plate'], as_index=False).first()
                .merge(chemlib, on='compound', how='left')
                .merge(n_genes, on='uniquecontrast', how='left'))
compounds_df = compounds_df[['gene', 'compound', 'plate', 'activity', 'n_genes',
                             'uniquecontrast', 'logfc', 'smiles']]
# MoleculeBatchID (per experiment, e.g. SRB-0004870-001) for the volcano label text,
# sourced from df_raw; the interface shows it in place of the bare compound id.
compounds_df['molecule_batch_id'] = compounds_df['uniquecontrast'].map(
    df_raw.drop_duplicates('uniquecontrast').set_index('uniquecontrast')['MoleculeBatchID'])
# drop Silent-activity experiments: no real down-modulation, removes the 'Silent'
# activity checkbox, and shrinks the embedded panel data (faster page load)
_n0 = len(compounds_df)
compounds_df = compounds_df[compounds_df['activity'] != 'Silent']
print(f'> dropped {_n0 - len(compounds_df):,} Silent-activity rows -> {len(compounds_df):,} remain')
print(f'> compounds_df: {len(compounds_df):,} (gene,compound,plate) rows across '
      f'{compounds_df["gene"].nunique():,} genes, {compounds_df["uniquecontrast"].nunique():,} volcanoes to render')

# --- colours + must-show + control genes + research, then render ---
DISEASE_AREA_COLORS = {
    'pharma': ACTIVE_C, 'BMS': '#BA09D9',
    'cancer or benign tumor': '#DD870E', 'hematologic disease': '#FF0000',
    'cardiovascular disease': "#FB008A", 'immune system disease': '#2A9D8F',
    'musculoskeletal or connective tissue disease': '#264653',
    'nervous system disease': '#963802', 'psychiatric disorder': '#000000',
    'nutritional or metabolic disease': '#17E804', 'endocrine system disease': "#6EE5F5",
}
MUST_INCLUDE = sorted(iface_df.loc[iface_df['disease_area'].isin(['pharma', 'BMS']), 'gene'])
# control compounds are no longer used to flag genes as controls (no grey
# diamonds); they're passed to the interface as a tickbox in the Activity
# panel — untick 'Control compounds' to remove them from every gene.
# gene_research = {gene_name: record}; tolerate a dict, a list of records, or a
# bad/stale value (-> empty; re-run the RESEARCH load cell to populate it).
_R = GENE_RESEARCH
if isinstance(_R, dict):
    gene_research = _R
elif isinstance(_R, (list, tuple)):
    gene_research = {r['gene_name']: r for r in _R if isinstance(r, dict) and 'gene_name' in r}
else:
    gene_research = {}
print(f'> gene_research: {len(gene_research)} genes' + ('' if gene_research else '  (empty - re-run the GENE_RESEARCH load cell)'))

fig, highlighted = fn.plot_3d_interface(
    iface_df,
    x_col='R2', y_col='association_score', z_col='ms_score',
    x_label='SAR predictability', y_label='association score', z_label='MS score',
    must_include=MUST_INCLUDE, top_n_highlight=40,
    compounds_df=compounds_df, volcano_source=meas, volcano_key='uniquecontrast', page_size=5,
    thumb_external=True,   # reference srb_png/<compound>.png next to the HTML (not inline base64)
    range_sliders=True, range_defaults={'x': 0.1, 'y': 0.5, 'z': 10}, # {SAR, OT, MS}
    activity_defaults=['Single', 'Low'],   # focus view: R2>0.1, assoc>0.6, MS>10 # 
    control_compounds=control_compounds, control_default_on=False,  # hide controls by default
    contaminant_compounds=contaminants, contaminant_default_on=False,  # hide contaminants by default
    gene_research=gene_research,
    validated_targets=validated_targets, devalidated_targets=devalidated_targets,  # Target validation (Y/N) tickboxes
    validated_label='FBXO31 dependent', devalidated_label='FBXO31 independent',
    depmap_defaults=['Selective', 'Non-essential'], conf_defaults=['High', 'Med'],
    lof_defaults=['Yes'], validation_defaults=[],  # default ticked boxes on load (validation: none)
    volcano_significant=True, volcano_dir=os.path.join(DROPBOX_ML, 'interfaces', 'volcanoes_px'),
    volcano_n_jobs=8, volcano_xlim=(-8, 8), volcano_size_px=350,
    disease_area_colors=DISEASE_AREA_COLORS, nb_display=False,
    html_path=os.path.join(GTLOCAL, 'interfaces', '20260612_3d_interface_PX_R2_assoc_ms.html'), #  # DROPBOX_ML
)

> iface_df: 4,640 gene dots (whole Px) | disease_area set for 4,513
> dropped 25 Silent-activity rows -> 43,597 remain
> compounds_df: 43,597 (gene,compound,plate) rows across 8,038 genes, 2,098 volcanoes to render
> gene_research: 6706 genes
> 4,639 / 4,639 genes after filtering (x=R2, y=association_score, z=ms_score)
  [highlight] corner-top-40=40, association_score>None: 0, must=25, union=63
  [range_sliders] panels built for all 4639 plotted genes


compound panels: 100%|██████████| 4345/4345 [00:22<00:00, 190.37gene/s]


> long-format panel: 22,087 compound entries across 4345 genes with compounds (page_size=5, 93 plates)
> built 22,087 structure thumbnails across 4639 genes (png=22087, rdkit=0, missing=0)
> volcanoes: 23,241 cached, 0 rendered [interactive SVG] -> /mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/4_Data_Sciences/15_ML/interfaces/volcanoes_px
  [range_sliders] default box ≈ 254 genes; gene labels shown while ≤ 274 in range (drag handles to widen/narrow)
  [activity_defaults] focus view starts on ['Low (2-10)', 'Single (1)']
wrote /mnt/c/Users/gtamo/Desktop/GT/interfaces/20260612_3d_interface_PX_R2_assoc_ms.html  (1.4 MB main doc)  +  20260612_3d_interface_PX_R2_assoc_ms_data.js  (19.0 MB, deferred)


In [ ]:
select_genes = highlighted[ (highlighted['ms_score']>10)  & (highlighted['R2']>0.1) & (highlighted['association_score']>0.35)]
select_genes[['gene']].to_csv('/mnt/c/Users/gtamo/Desktop/GT/disease_association/20260611_select_genes.csv',index=False,sep=',')

In [32]:
iface_df[iface_df['gene'].isin(['KAT2A','KAT2B'])]

,gene,ms_score,R2,association_score,disease_area
1924,KAT2A,5.476685,0.016021,0.629283,cancer or benign tumor
1925,KAT2B,2.886401,0.019239,0.646822,cancer or benign tumor


In [26]:
df_raw[df_raw['genes']=='KATB']

,MoleculeBatchID,MSPlate,genes,pg,logfc,pvalue,adjpval,significant,uniquecontrast,compound,batch,-log10(p-value),ms_score


## 1. Build interface on restricted set of genes

In [ ]:
## local params:
control_compounds = ['SRB-0000692','SRB-0000615']

In [ ]:
## degradation research per target (one record/gene): therapeutic rationale,
## degrader feasibility, DepMap dependency, indications, safety, sources, ...
import json as _json
with open('/mnt/c/Users/gtamo/Desktop/GT/disease_association/20260604_degradation_research.json') as _f:
    GENE_RESEARCH = _json.load(_f)
print(f'> loaded degradation research for {len(GENE_RESEARCH)} genes; '
      f'fields: {list(GENE_RESEARCH[0].keys())}')


In [ ]:
## ============================================================
## 3D interface: R² × association_score × ms_score  (fn.plot_3d_interface)
##   • dots          = FBX_MSSCORE (one per gene), coloured by disease area
##                     with pharma-patent + BMS-pipeline overrides
##   • volcano source = FBX_MEASURE (per gene×experiment logfc/pvalue)
##   • compounds      = linked via FBX_REPORT.uniquecontrast → srbnumber
## For each target we list EVERY compound that significantly down-modulates it
## (logfc ≤ -1 & p ≤ 0.05 in any plate replicate), keeping the strongest-hit
## experiment for the volcano. The panel pages 5 compounds at a time (◀ ▶).
## ============================================================
DROP_PLATES = ['Plate12', 'Plate15', 'Plate23']

# chem library — SMILES for the structure thumbnails (key = SRB-XXXXXXX, no batch)
chemlib = pd.read_csv(CHEMLIB_PATH)[['compound', 'smiles']].drop_duplicates('compound')

# --- 1) highest-priority OpenTargets disease area per gene ---
if 'ot_df' not in globals():
    ot_df = pd.read_parquet(OT_CACHE)
PRIORITY = [
    'cancer or benign tumor', 'hematologic disease', 'cardiovascular disease',
    'immune system disease', 'musculoskeletal or connective tissue disease',
    'nervous system disease', 'psychiatric disorder',
    'nutritional or metabolic disease', 'endocrine system disease',
]
_rank = {a: i for i, a in enumerate(PRIORITY)}
_areas = (ot_df[['target_symbol', 'overall_score', 'disease_name', 'therapeutic_areas']]
          .assign(area=lambda d: d['therapeutic_areas'].fillna('').str.split('|'))
          .explode('area'))
_areas = _areas[_areas['area'].isin(_rank)].copy()
_areas['_rank'] = _areas['area'].map(_rank)
_top_area = (_areas.sort_values(['target_symbol', '_rank', 'overall_score'],
                                ascending=[True, True, False])
             .drop_duplicates('target_symbol', keep='first')
             .rename(columns={'target_symbol': 'gene', 'area': 'disease_area'})
             [['gene', 'disease_area']])

# --- 2) dots: FBX_MSSCORE + disease_area + pharma/BMS override (BMS supersedes) ---
iface_df = FBX_MSSCORE.rename(columns={'genes': 'gene'}).merge(_top_area, on='gene', how='left')
_pharma = set(pd.read_csv(PHARMA_PATENT_CSV)['gene'].dropna().unique())
_bms    = set(pd.read_csv(BMS_GENES)['hgnc_symbol'].dropna().unique())
_model  = iface_df['R2'] > PHARMA_R2_CUTOFF
iface_df.loc[iface_df['gene'].isin(_pharma) & _model, 'disease_area'] = 'pharma'
iface_df.loc[iface_df['gene'].isin(_bms)    & _model, 'disease_area'] = 'BMS'
print(f'> iface_df: {len(iface_df)} genes; '
      f'disease_area set for {iface_df["disease_area"].notna().sum()}')

# --- 3) per-target compound list from FBX_MEASURE + FBX_REPORT ---
meas = FBX_MEASURE[~FBX_MEASURE['plate'].isin(DROP_PLATES)]            # volcano source
# genes measured per experiment (matches the count plot_volcano used to show in its title)
n_genes = (meas.dropna(subset=['logfc', 'pvalue'])
           .groupby('uniquecontrast')['genes'].nunique().rename('n_genes'))
rep  = (FBX_REPORT[['uniquecontrast', 'srbnumber', 'plate', 'concentration', 'activity']]
        .drop_duplicates('uniquecontrast'))
_p = rep['srbnumber'].astype(str).str.split('-', n=2, expand=True)     # strip batch suffix
rep = rep.assign(compound=_p[0] + '-' + _p[1])                         # SRB-XXXXXXX (chemlib key)
_hit = meas.loc[(meas['significant'] == 1) & (meas['logfc'] < 0),
                ['genes', 'uniquecontrast', 'logfc', 'pvalue']]   # significantly DOWN-modulated (FBX flag); drop meas 'plate'
hits = _hit.merge(rep, on='uniquecontrast', how='left').rename(columns={'genes': 'gene'})
hits = hits[hits['compound'].notna() & hits['compound'].str.startswith('SRB-')]
# Keep ALL passing plates (one row per gene,compound,plate) so the panel can
# offer per-plate volcanoes + checkbox filtering. One row per (gene,compound,plate):
# strongest-hit uniquecontrast within that plate.
hits = hits.sort_values(['gene', 'compound', 'plate', 'logfc'])
compounds_df = (hits.groupby(['gene', 'compound', 'plate'], as_index=False).first()
                .merge(chemlib, on='compound', how='left')
                .merge(n_genes, on='uniquecontrast', how='left'))
compounds_df = compounds_df[['gene', 'compound', 'plate', 'activity', 'n_genes',
                             'uniquecontrast', 'logfc', 'smiles']]
_per_cmp = compounds_df.groupby(['gene', 'compound']).size()
print(f'> {len(compounds_df):,} (gene,compound,plate) rows; '
      f'{_per_cmp.shape[0]:,} (gene,compound) pairs across '
      f'{compounds_df["gene"].nunique():,} genes; '
      f'{int((_per_cmp > 1).sum()):,} pairs span >1 plate; '
      f'plates: {sorted(compounds_df["plate"].unique())}')

# --- 4) colours + must-show pharma/BMS, then render ---
DISEASE_AREA_COLORS = {
    'pharma': ACTIVE_C, 'BMS': '#BA09D9',
    'cancer or benign tumor': '#DD870E', 'hematologic disease': '#FF0000',
    'cardiovascular disease': "#FB008A", 'immune system disease': '#2A9D8F',
    'musculoskeletal or connective tissue disease': '#264653',
    'nervous system disease': '#963802', 'psychiatric disorder': '#000000',
    'nutritional or metabolic disease': '#17E804',
    'endocrine system disease': "#6EE5F5",
}
MUST_INCLUDE = sorted(iface_df.loc[iface_df['disease_area'].isin(['pharma', 'BMS']), 'gene'])

# control targets: genes whose significant-down compound(s) are ALL control compounds
_g2c = compounds_df.groupby('gene')['compound'].apply(set)
control_genes = sorted(g for g, cs in _g2c.items() if cs & set(control_compounds))
print(f'> {len(control_genes)} control targets (with >=1 control compound): {control_genes[:10]}{"..." if len(control_genes)>10 else ""}')

# per-gene degradation research (gene_name -> record) for the hover research box
gene_research = {r['gene_name']: r for r in GENE_RESEARCH}

fig, highlighted = fn.plot_3d_interface(
    iface_df,
    x_col='R2', y_col='association_score', z_col='ms_score',
    x_label='SAR predictability',
    y_label='association score',
    z_label='MS score',
    must_include=MUST_INCLUDE,
    top_n_highlight=40,
    compounds_df=compounds_df,           # all associated compounds, paginated
    volcano_source=meas,                 # FBX_MEASURE = volcano data
    volcano_key='uniquecontrast',
    page_size=5,
    range_sliders=True,            # R²/association/MS-score range gating (in-range = coloured)
    range_defaults={'y': 0.35},    # default association (y) lower handle at 0.35
    control_genes=control_genes,   # grey diamonds, legend 'control'
    gene_research=gene_research,   # per-gene degradation-research hover box
    volcano_significant=True,      # colour only significant targets in the volcanoes
    volcano_dir=os.path.join(DROPBOX_ML, 'interfaces', 'volcanoes'),  # cache PNGs next to the HTML; re-runs reuse them
    volcano_n_jobs=8, volcano_xlim=(-8, 8), volcano_size_px=350,
    disease_area_colors=DISEASE_AREA_COLORS,
    nb_display=False,
    html_path=os.path.join(DROPBOX_ML,'interfaces', '20260603_3d_interface_R2_assoc_ms.html'),
)


In [ ]:
## extract the list of relevant targets: 
FBX_MSSCORE[['genes']].to_csv('output/interface/20260604_genes_poc.csv',index=False,sep=',')

## 2. Get cell signature

In [ ]:
## Gene-function explanation from the local cell-signature annotations
## (built by python/build_cell_signature_annotations.py -> output/cell_signature/).
## protein_function = UniProt name/features; gene2term = GO BP/MF/CC + Reactome.
_FUNC = pd.read_parquet('output/cell_signature/protein_function.parquet').set_index('gene')
_G2T  = pd.read_parquet('output/cell_signature/gene2term.parquet')
_TSIZE = _G2T.groupby('term_id')['gene'].nunique()   # term breadth -> rank specific first

def explain_gene(gene, n=6):
    """Concise functional summary for a gene: protein name + UniProt features +
    the most-specific GO / Reactome terms (GO is propagated, so we sort by term
    breadth to surface informative leaves over broad ancestors)."""
    print(f'=== {gene} ===')
    if gene in _FUNC.index:
        r = _FUNC.loc[gene]
        tag = ', reviewed' if r.reviewed else ''
        print(f'{r.protein_name}  (UniProt {r.uniprot}{tag})')
        feats = [lbl for lbl, flag in [
            ('transmembrane', r.transmembrane), ('signal peptide', r.signal_peptide),
            ('DNA-binding', r.dna_binding), ('nucleotide-binding', r.nucleotide_binding),
            ('zinc finger', r.zinc_finger), ('catalytic site', r.catalytic)] if flag]
        if feats:    print('  features:', ', '.join(feats))
        if r.pfam:   print('  Pfam:', r.pfam)
        if r.domains:print('  domains:', r.domains)
    else:
        print('  (no protein-function annotation)')
    g = (_G2T[_G2T['gene'] == gene]
         .assign(_sz=lambda d: d['term_id'].map(_TSIZE))
         .sort_values('_sz'))   # most-specific terms first
    for coll, label in [('GO_BP', 'GO process'), ('GO_MF', 'GO function'),
                        ('GO_CC', 'GO location'), ('Reactome', 'Reactome')]:
        terms = g.loc[g['collection'] == coll, 'term_name'].tolist()
        if terms:
            more = ' ...' if len(terms) > n else ''
            print(f'  {label} ({len(terms)}): ' + '; '.join(terms[:n]) + more)

explain_gene('RAD51')


In [ ]:
## ── Volcano coloured by protein FUNCTION (not just up/down) ───────────────
## Same significant-gene volcano as in the 3D interface, but every significant
## protein is coloured by its coarse functional category (fn.categorize_genes,
## derived from the GO/Reactome annotations) and DIRECTION is shown by marker
## shape: ▲ up-modulated, ▼ down-modulated. Lets you read the cell response off
## the plot — e.g. a down-cluster of DNA-replication / chromatin / translation.
## Example: compound SRB-0005654, target LOX (lysyl oxidase, ECM / adhesion).
gene_category = fn.categorize_genes(_G2T)        # {gene: functional category}

_meas = FBX_MEASURE[~FBX_MEASURE['plate'].isin(DROP_PLATES)]
_rep  = FBX_REPORT[['uniquecontrast', 'srbnumber', 'plate']].drop_duplicates('uniquecontrast')
_p    = _rep['srbnumber'].astype(str).str.split('-', n=2, expand=True)
_rep  = _rep.assign(compound=_p[0] + '-' + _p[1])               # SRB-XXXXXXX (strip batch)
_uc   = _rep.loc[(_rep['compound'] == 'SRB-0005654') & (_rep['plate'] == 'Pw82KO'),
                 'uniquecontrast'].iloc[0]

fig, ax = plt.subplots(figsize=(8, 6), dpi=110)
_agg = fn.plot_volcano_significant(
    _meas, _uc, 'LOX',
    gene_category=gene_category, category_colors=fn.CATEGORY_COLORS,
    xmin=-8, xmax=8, ax=ax,
    title='SRB-0005654 · LOX — significant proteins coloured by function')
plt.show()


In [ ]:
## ── Qualify the cell response: ORA (hypergeometric) + GSEA-preranked ──────
## Two complementary enrichment tests on the SAME volcano (_uc above):
##   • ORA  — hypergeometric (Fisher) over the THRESHOLDED significant set
##            (down / up separately), measured proteome as background, BH-FDR.
##            Maps 1:1 onto the coloured points in the volcano above.
##   • GSEA — threshold-free: ranks ALL measured proteins by
##            sign(logfc)·-log10(p) and tests concentration at top/bottom.
##            Catches coordinated subtle shifts no single gene clears the cutoff for.
## Processes called by BOTH tests are the trustworthy story.
_sub = _meas[_meas['uniquecontrast'] == _uc].dropna(subset=['genes', 'logfc', 'pvalue'])
_a   = _sub.groupby('genes').agg(logfc=('logfc', 'mean'), pvalue=('pvalue', 'min'),
                                 sig=('significant', 'max')).reset_index()
_bg   = set(_a['genes'])
_down = set(_a.loc[(_a.sig > 0) & (_a.logfc < 0), 'genes'])
_up   = set(_a.loc[(_a.sig > 0) & (_a.logfc > 0), 'genes'])
_ranks = pd.Series((np.sign(_a.logfc) * -np.log10(_a.pvalue.clip(lower=1e-300))).values,
                   index=_a['genes'])

ora_down = fn.ora_enrichment(_down, _bg, _G2T, fdr=0.05, top_n=10)
ora_up   = fn.ora_enrichment(_up,   _bg, _G2T, fdr=0.05, top_n=10)
gsea     = fn.gsea_preranked(_ranks, _G2T, n_perm=1000, seed=0)   # ~1 min (permutation null)

def _show(title, df, cols):
    print(title)
    print(df[cols].to_string(index=False) if len(df) else '  (none)'); print()

print(f'SRB-0005654 · Pw82KO  —  measured {len(_bg)}, down {len(_down)}, up {len(_up)}\n')
_show('ORA  ▼ DOWN  (hypergeometric, FDR<0.05)', ora_down, ['collection', 'term_name', 'k', 'K', 'fdr'])
_show('ORA  ▲ UP    (hypergeometric, FDR<0.05)', ora_up,   ['collection', 'term_name', 'k', 'K', 'fdr'])
_show('GSEA ▼ suppressed (NES<0, FDR<0.05)',
      gsea[(gsea.direction == 'down') & (gsea.fdr < 0.05)].head(10), ['collection', 'term_name', 'NES', 'fdr'])
_show('GSEA ▲ induced    (NES>0, FDR<0.05)',
      gsea[(gsea.direction == 'up') & (gsea.fdr < 0.05)].head(10), ['collection', 'term_name', 'NES', 'fdr'])


In [ ]:
## ── Enrichment score per FUNCTION (the 15 categories as gene sets) ────────
## Roll the per-GO/Reactome enrichment up to the coarse functions: feed the
## {gene: function} map (gene_category, from the volcano cell above) as gene
## sets to BOTH tests, so each FUNCTION gets one ORA p/FDR and one GSEA NES/FDR.
## Categories are large sets -> relax the size caps.
g2cat   = fn.gene_category_long(gene_category)
ora_fn  = fn.ora_enrichment(_down, _bg, g2cat, collections=('Function',),
                            min_overlap=3, max_term_size=10**9)
gsea_fn = fn.gsea_preranked(_ranks, g2cat, collections=('Function',),
                            min_size=5, max_size=10**9, n_perm=1000)

func_enrich = (gsea_fn[['term_name', 'size', 'NES', 'fdr', 'direction']]
               .rename(columns={'term_name': 'function', 'fdr': 'gsea_fdr'})
               .merge(ora_fn[['term_name', 'k', 'K', 'fdr']]
                      .rename(columns={'term_name': 'function', 'k': 'down_n',
                                       'K': 'category_size', 'fdr': 'ora_down_fdr'}),
                      on='function', how='left')
               .sort_values('NES'))
print(f'SRB-0005654 · Pw82KO — enrichment per function (GSEA NES + ORA-down FDR):')
func_enrich[func_enrich['function'] != 'Other'].reset_index(drop=True)


## Build interface on full set of genes